# Periodic Point Vortex Model
This is a simple Python code for solving the periodic point vortex model (PVM) based on the model described in *Weiss and McWilliams, nonergodicity of point vortices, Phys. Fluids A 3 (5), May 1991*.

The code reads in an initial state from a file located in *./data/vortex.[fileNumber]* that consists of three columns of data: *x-position, y-position, circulation*

*./data/curframe.dat* is read in at the start of the simulation and includes values for *currentTime* and *fileNumber*. If the latter is non-negative, that corresponding file *./data/vortex.[fileNumber]* will be read in as an initial condition. If *fileNumber* is negative, a random array of vortices are produced as an initial condition.

The code records the values of the : *Time, Hamiltonian, x-Momentum, y-Momentum* in files *./data/hamiltonian.[fileNumber]* (Note: the Hamiltonian is only conserved if vortices are not removed/re-inserted and the Momentum is conserved up to *\pm 2\pi* for each vortex passing through the periodic boundaries)

The PVM is solved using an adaptive Runge-Kutta Cash-Karp Method and includes a novel dissipation scheme whereby nearest vortices of opposite sign (vortex dipoles) are removed if too close (*< removalDistanceLower*) or too far (*> removalDistanceUpper*) and reinserted randomly into the domain as a dipole of size *interVortexDistance*.

In [514]:
# IMPORT REQUIRED PYTHON LIBRARIES
import numpy as np

In [515]:
# DEFINE GLOBAL VARIABLES

Np = 6                 # number of positive vortices
Nm = 6                 # number of negative vortices
N = Np + Nm            # total number of vortices 

totalSteps = 100000    # total time steps 
dt = 1.e-3             # start time step
dtMax = 1.e-2          # max time step used in adaptive algorithm
rkTolerance = 1.e-4    # set the Runge-Kutta tolerance between the 4th and 5th order approximations

outputTime = 0.5       # output data when comptuational time are multiples of this value
   
Lx = 2.0 * np.pi       # x-domain length for periodic PVM
Ly = 2.0 * np.pi       # y-domain length for periodic PVM
imageTotal = 5         # number of images used in the periodic PVM ( = 5 chosed based on experience)
interVortexDistance = np.sqrt(Lx*Ly/N) # mean intervortex distance (average distance between any two vortices)

FLAG_DISSIPATION = True # turns on code that removes/reinserrs dipoles that are closer than  < removalDistanceLower  or further > removeralDistanceUpper

removalDistanceLower = 0.2 * interVortexDistance    # if dipoles are closer than this, then they are removed and reinserted
removalDistanceUpper = 5.0 * interVortexDistance    # if nearest opposite signed vortex is larger than this distance then the two vortices ("dipole") are removed and reinserted

In [516]:
# DECLARE VARIABLES AND ARRAYS

timeCurrent = 0.0       # current time
fileNumber = 0          # current file number (integer value that is incremented every outputTime)

xy = np.zeros((N,2))    # array for vortex position
g = np.zeros(N)         # array for vortex circulations

In [517]:
# DEFINE INITIAL VORTEX STATE
# code reads in data located in ./data/curframe.dat which is the last values of currentTime and fileNumber
# Based on this, the function will either produce a random intial state or load previous state from file ./data/vortex.<fileNumber>  

def initialState(xy,g,timeCurrent,fileNumber):

    timeCurrent, fileNumber = np.loadtxt("./data/curframe.dat")         # load timeCurrent and fileNumber from ./data/curframe.dat
    fileNumber = int(fileNumber)                                        # ensure that fileNumber is an integer    
    
    if fileNumber < 0:                                                  # if fileNumber is negative then produce random state and set timeCurrent and fileNumber both to zero
        for index in range(0,Np):
            xy[index,:] = [Lx*np.random.rand(),Ly*np.random.rand()]     # random position, uniformly distributed between 0 and 2\pi
            g[index] = 1.0/N                                            # first Np vortices in array will have positive circulation of 1/N
        
        for index in range(Np,N):
            xy[index,:] = [Lx*np.random.rand(),Ly*np.random.rand()]     # random position, uniformly distributed between 0 and 2\pi
            g[index] = -1.0/N                                           # last Nm vortices in array will have negative circulation of -1/N
        
    
        timeCurrent = 0.0                                               #set timeCurrent and fileNumber to zero
        fileNumber = 0

        np.savetxt("./data/curframe.dat", np.column_stack((timeCurrent, fileNumber)), delimiter = ' ',fmt = '%1.12e %i')    # record timeCurrent and fileNumber in ./data/curframe.dat
        np.savetxt("./data/vortex.%.6d" % (fileNumber), np.column_stack((xy,g)), delimiter=' ',fmt = '%1.12e')          # record intial state in file ./data/vortex.000000
        
    else:               # if fileNumber is non-negative then load in data from pre-existing file ./data/vortex.<fileNumber>
        
        xy = np.loadtxt("./data/vortex.%.6d" % (fileNumber),usecols=(0,1))         # read vortex position data from file
        g = np.loadtxt("./data/vortex.%.6d" % (fileNumber),usecols=(2))            # read vortex circulation data from file
    
    return xy, g, timeCurrent,fileNumber                                               # return all new variables and arrays back to main function


In [518]:
# COMPUTE THE VELOCITY FIELD OF PVM    

def computeVelocity(xy):
    
    dv = np.zeros((N,2))                        # define an empty array for velocities


    # compute velocity for PERIODIC PVM
    for iIndex in range(0,N):               # for each vortex i, velocity is computed by summing over all remaining N-1 vortices
        for jIndex in range(0,N):
            if jIndex != iIndex:            # make sure that vortex i and vortex j are not the same
                xij = xy[iIndex,0]- xy[jIndex,0]
                yij = xy[iIndex,1]- xy[jIndex,1]
                    
                dv[iIndex,:] += [-(0.25/np.pi)* g[jIndex] * np.sin(yij)/ (np.cosh(xij)-np.cos(yij)), (0.25/np.pi)* g[jIndex] * np.sin(xij)/ (np.cosh(yij)-np.cos(xij))] # velocity formula for periodic PVM in original box

                for nImage in range(1,imageTotal): # velocity formula for periodic PVM in the periodic images (first 5 images have been shown to be sufficient for convergence)
                    dv[iIndex,:] += [-(0.25/np.pi)* g[jIndex] * np.sin(yij)/ (np.cosh(xij-2.0*np.pi*nImage)-np.cos(yij))-(0.25/np.pi)* g[jIndex] * np.sin(yij)/ (np.cosh(xij+2.0*np.pi*nImage)-np.cos(yij)), (0.25/np.pi)* g[jIndex] * np.sin(xij)/ (np.cosh(yij-2.0*np.pi*nImage)-np.cos(xij))+(0.25/np.pi)* g[jIndex] * np.sin(xij)/ (np.cosh(yij+2.0*np.pi*nImage)-np.cos(xij))]
    return dv
   
    

In [519]:
# INVOKE PERIODIC BOUNDARY CONDITIONS
# code checks (after every time step) to see if vortex has moved passed the original periodic domain. If it has then the vortex is re-wrapped back into the original box with a 2\pi shift.

def invokeBoundaryConditions(xy):

 
    for iIndex in range(0,N):       # iterate through all vorticies and check to see if x-position is greater than Lx or less than zero, and then adjust accordingly
        if xy[iIndex,0] >= Lx:
            xy[iIndex,0] -= Lx
        elif xy[iIndex,0] < 0.0:
            xy[iIndex,0] += Lx
        
        if xy[iIndex,1] >= Ly:      # iterate through all vorticies and check to see if y-position is greater than Lx or less than zero, and then adjust accordingly
            xy[iIndex,1] -= Ly
        elif xy[iIndex,1] < 0.0:
            xy[iIndex,1] += Ly

    return


In [520]:
# 4TH/5TH ORDER ADAPTIVE RUNGE-KUTTA CASH-KARP METHOD

def rungeKutta45(xy,dt):
  
  k1 = dt * computeVelocity(xy)
  k2 = dt * computeVelocity(xy + (k1/5.0))
  k3 = dt * computeVelocity(xy + (3.0/40.0)*k1 + (9.0/40.0)*k2 )
  k4 = dt * computeVelocity(xy + (3.0/10.0)*k1 - (9.0/10.0)*k2 + (6.0/5.0)*k3)
  k5 = dt * computeVelocity(xy - (11.0/54.0)*k1 + (5.0/2.0)*k2 - (70.0/27.0)*k3 + (35.0/27.0)*k4)
  k6 = dt * computeVelocity(xy + (1631.0/55296.0)*k1 + (175.0/512.0)*k2 + (575.0/13824.0)*k3 + (44275.0/110592.0)*k4 + (253.0/4096.0)*k5)

  sol4 = xy + (37.0/378.0)*k1 + (250.0/612.0)*k3 + (125.0/594.0)*k4 + (512.0/1771.0)*k6                                 # 4th order approximation
  sol5 = xy + (2825.0/27648.0)*k1 + (18575.0/48384.0)*k3 + (13525.0/55296.0)*k4 + (277.0/14336.0)*k5 + (1.0/4.0)*k6     # 5th order approximation

  # ADAPTIVE STEP

  errorSol = np.abs(sol4 - sol5).max()                                                              # compute error between 4th and 5th order approximations
                                                                                     
  if errorSol > 1.e12:                                                                              # if error is extremely large then exit code with warning
	  exit("Error in timestepping routine is too large....code aborted")
  
  elif errorSol < rkTolerance:                                                                      # error is within tolerance, solution is good, take 5th order solution
	  dtNew = min( 0.9 * dt * abs(rkTolerance/errorSol)**0.25, 5.0*dt, dtMax)                         # adapt time step to be slightly larger if possible
	  return sol5, dtNew                                                                              # return solution and new time step
  
  else:                                                                                             # error is above tolerance, will recursively call rungeKutta45() again but with smaller time step                 
    dtNew = 0.9 * dt * abs(rkTolerance/errorSol)**0.2                                               # estimate of smaller time step                                     
    xy, dt = rungeKutta45(xy,max(dtNew, 0.1*dt))                                                    # apply rungeKutta45() again
    return xy, dt


In [521]:
# COMPUTE HAMILTONIAN OF THE POINT VORTEX SYSTEM

def computeHamiltonian(xy,g):
    Value = 0.0          # initilise hamiltonian value
   
    # PERIODIC PVM
   
    for iIndex in range(0,N):               # double summation of vortex array
        for jIndex in range(iIndex+1,N):
            
            xij = xy[iIndex,0]- xy[jIndex,0]
            yij = xy[iIndex,1]- xy[jIndex,1]
                   
            Value -= (0.25/np.pi) * g[iIndex] * g[jIndex] * np.log(np.cosh(xij)-np.cos(yij))       # compute hamiltonian

            for nImage in range(1,imageTotal):       # compute hamiltonian over periodic images
                Value -= (0.25/np.pi) * g[iIndex] * g[jIndex] * np.log(1.0 + (np.sinh(xij)*np.sinh(xij)*(1.0 - np.tanh(2.0*np.pi*nImage)*np.tanh(2.0*np.pi*nImage)) ) + (np.cos(yij)*np.cos(yij)/(np.cosh(2.0*np.pi*nImage)*np.cosh(2.0*np.pi*nImage))) - (2.0*np.cosh(xij)*np.cos(yij)/np.cosh(2.0*np.pi*nImage)))
                    

            Value += (0.125/np.pi**2) * g[iIndex] * g[jIndex] * xij**2    # compute hamiltonian
    return Value          


In [522]:
# COMPUTE MOMENTUM OF THE POINT VORTEX SYSTEM

def computeMomentum(xy,g):

    XValue = 0.0            # initilise x-momentum value
    YValue = 0.0            # initilise x-momentum value
    
    XValue = np.sum(g * xy[:,0])        # compute x-momentum value
    YValue = np.sum(g * xy[:,1])        # compute y-momentum value

    return XValue, YValue          


In [523]:
# SUBROUTINES THAT REMOVES OPPOSITE SIGNED VORTICES IF TOO CLOSE/FAR AND RE-INSERTS THEM RANDOMLY AT INTER-VORTEX SCALE PART


# DEFINE A DISTANCE FUNCTION    
def distance(xy,i,j):
    dx = xy[i,0] - xy[j,0]  # x-distance between vortex i and vortex j 
    dy = xy[i,1] - xy[j,1]  # y-distance between vortex i and vortex j 

    if dx >= 0.5*Lx:        # adjusts for periodic boundaries in x direction
        dx -= Lx
    elif dx < -0.5*Lx:
        dx += Lx

    if dy >= 0.5*Ly:        # adjusts for periodic boundaries in y direction
        dy -= Ly
    elif dy < -0.5*Ly:
        dy += Ly

    return np.sqrt(dx**2 + dy**2)      # returns (shortest) distance between both vortices

# A FUNCTION THAT DETERMINES NEAREST VORTEX OF OPPOSTITE SIGN FOR EACH VORTEX IN ARRAY
def nearestNeighbour(xy,candidateForDeletion,neighbour,candidateDistance):
    
    for iIndex in range(0,Np):                              # for positively circulated vorticies only
        if candidateForDeletion[iIndex] == 0:               # if vortex iIndex is not a candidate for deletion
            distanceClosest = 2*(Lx+Ly)                     # set very long closest distance that will be minimised
            for jIndex in range(Np,N):                      # cycle through all negatively signed vortices and determine the closest one
                if candidateForDeletion[jIndex] == 0:       # check to see if vortex jIndex is not already being deleted
                    dist = distance(xy,iIndex,jIndex)       # compute distance 
                    if dist < distanceClosest:              # if new closest opposite-signed vortex store distance and index
                        distanceClosest = dist
                        jClosest = jIndex
            neighbour[iIndex] = jClosest         
            candidateDistance[iIndex] = distanceClosest

    for iIndex in range(Np,N):                              # for positively circulated vorticies only
        if candidateForDeletion[iIndex] == 0:               # if vortex iIndex is not a candidate for deletion
            distanceClosest = 2*(Lx+Ly)                     # set very long closest distance that will be minimised
            for jIndex in range(0,Np):                      # cycle through all negatively signed vortices and determine the closest one   
                if candidateForDeletion[jIndex] == 0:       # check to see if vortex jIndex is not already being deleted
                    dist = distance(xy,iIndex,jIndex)       # compute distance
                    if dist < distanceClosest:              # if new closest opposite-signed vortex store distance and index
                        distanceClosest = dist
                        jClosest = jIndex
            neighbour[iIndex] = jClosest
            candidateDistance[iIndex] = distanceClosest
           

    FLAG_Recursion = False                                  # set default state of recursive flag to be False

    for iIndex in range(0,N):
        if candidateForDeletion[iIndex] == 0:
            if candidateDistance[iIndex] < removalDistanceLower:
                if (iIndex == neighbour[neighbour[iIndex]]):              # if both vortices see each other as nearest neighbours
                    candidateForDeletion[iIndex] = -1                     # sets vortex iIndex for deletion, -1 to show it's due to < removalDistanceLower                       
                else:
                    FLAG_Recursion = True                                 # if the two neighbours don't agree, we will recursively apply nearestNeighbour function
            elif candidateDistance[iIndex] > removalDistanceUpper:
                if(iIndex == neighbour[neighbour[iIndex]]):               # if both vortices see each other as nearest neighbours
                    candidateForDeletion[iIndex] = 1                      # sets vortex iIndex for deletion, +1 to show it's due to > removalDistanceUpper
                else:
                    FLAG_Recursion = True                                 # if the two neighbours don't agree, we will recursively apply nearestNeighbour function
    if FLAG_Recursion == True:                               
        nearestNeighbour(xy,candidateForDeletion,neighbour,candidateDistance)      # recursively apply nearestNeighbour function
    else:
        return

# REMOVES SELECTED VORTICES AND REINSERTS A DIPOLE (OF INTER-VORTEX DISTANCE SIZE) BACK INTO THE BOX AT RANDOM POSITION 
def removeVortices(xy,g):

    candidateForDeletion = np.zeros(N,np.int64)                 # vector declarations for identifying if vortex is deleted
    neighbour = np.zeros(N,np.int16)                            # vector detailing which vortex is nearest neighbour
    candidateDistance = np.zeros(N)                             # distance of nearest neighbour

    nearestNeighbour(xy,candidateForDeletion,neighbour,candidateDistance)          # find nearest neighbout
   
    for iIndex in range(0,Np):                                         # as vortices will be removed in pair, only consider positive signed vortices
        if candidateForDeletion[iIndex] != 0:                           # all vortices identifed for removal
            xy[iIndex,:] = [Lx*np.random.rand(),Ly*np.random.rand()]    # reposition vortex and its nearest neighbour
            phase = 2.0*np.pi*np.random.rand()
            xy[neighbour[iIndex],:] = [xy[iIndex,0] + interVortexDistance*np.cos(phase),xy[iIndex,1] + interVortexDistance*np.sin(phase)]

            if xy[neighbour[iIndex],0] >= Lx:                           # check to see if nearest neighbour has gone over periofic boundaries
                xy[neighbour[iIndex],0] -= Lx
            elif xy[neighbour[iIndex],0] < 0.0:
                xy[neighbour[iIndex],0] += Lx
        
            if xy[neighbour[iIndex],1] >= Ly:                              
                xy[neighbour[iIndex],1] -= Ly
            elif xy[neighbour[iIndex],1] < 0.0:
                xy[neighbour[iIndex],1] += Ly

            candidateForDeletion[iIndex] = 0                            # reset delcaration for removal
            candidateForDeletion[neighbour[iIndex]] = 0

    return

In [524]:
############################################

# MAIN PYTHON CODE FOR PVM

############################################



xy, g, timeCurrent, fileNumber = initialState(xy,g,timeCurrent,fileNumber)          # get initial state

for stepNumber in range(1,totalSteps+1):                                              # start time stepping loop
   
    xy, dt = rungeKutta45(xy,dt)                                                    # time step using rungeKutta45
    
    invokeBoundaryConditions(xy)                                                    # invoke boundary conditions
    
    if FLAG_DISSIPATION == True:
        removeVortices(xy,g)                                                        # redistirbutes closest dipole vortices if too close or too far
    
    timeCurrent += dt                                                               # increment timeCurrent
   
    # RECORD DATA   
    if int(timeCurrent / outputTime) == fileNumber +1:                              # record data every outputTime            
        fileNumber +=  1                                                            # incremenet fileNumber by one
    
        hamiltonianValue = computeHamiltonian(xy,g)                                 # compute hamiltonian 

        momentumXValue, momentumYValue = computeMomentum(xy,g)                      # compute momentum 
        
        np.savetxt("./data/hamiltonian.%.6d" % (fileNumber), np.column_stack((timeCurrent, hamiltonianValue, momentumXValue, momentumYValue)), delimiter=' ', fmt='%1.12e')  # record hamiltonian and momentum values to ./data/hamiltonian.<fileNumber>

        np.savetxt("./data/vortex.%.6d" % (fileNumber), np.column_stack((xy, g)), delimiter=' ',fmt = '%1.12e')         # save vortex data to new file ./data/vortex.<fileNumber>
        np.savetxt("./data/curframe.dat",np.column_stack((timeCurrent, fileNumber)), delimiter = ' ',fmt = '%1.12e %i')     # updates ./data/curframe.dat
    
        print( "fileNumber = ", fileNumber," time = ", "%.6f" % timeCurrent, " dt = ", "%.6f" % dt, " hamiltonian = ", "%.6f" % hamiltonianValue, " momentum x = ", "%.6f" % momentumXValue, " momentum x = ", "%.6f" % momentumYValue)   # print some information to console 


fileNumber =  1  time =  0.50405  dt =  0.01000000  hamiltonian =  0.00857  momentum x =  -1.29009  momentum x =  -0.94500
fileNumber =  2  time =  1.00405  dt =  0.01000000  hamiltonian =  0.00857  momentum x =  -1.29009  momentum x =  -0.94500
fileNumber =  3  time =  1.50405  dt =  0.01000000  hamiltonian =  0.00857  momentum x =  -1.29009  momentum x =  -0.94500
fileNumber =  4  time =  2.00405  dt =  0.01000000  hamiltonian =  0.00857  momentum x =  -1.29009  momentum x =  -0.94500
fileNumber =  5  time =  2.50405  dt =  0.01000000  hamiltonian =  0.00857  momentum x =  -1.29009  momentum x =  -0.94500
fileNumber =  6  time =  3.00405  dt =  0.01000000  hamiltonian =  0.00857  momentum x =  -1.29009  momentum x =  -0.94500


KeyboardInterrupt: 